In [10]:
import numpy as np
import h5py
import seisbench.data as sbd
import matplotlib.pyplot as plt
import pandas as pd
import csv
import os

In [11]:
data = np.load('C:\\Users\\Administrator\\Desktop\\EV00001_AVO.npz')

In [12]:
print(data['data'].shape)
print(data['itp'])
print(data['channels'])
print(data['station_id'])

(9001, 3)
3000
E_N_U
AVO


In [13]:
h5path = 'D:\\seisbench_data\\vcseis\\'
path_list = []
path_list.append(h5path + 'ak_lp')
path_list.append(h5path + 'ak_ns')
path_list.append(h5path + 'ak_vt')
path_list.append(h5path + 'cas_lp')
path_list.append(h5path + 'cas_vt')
path_list.append(h5path + 'hw_vt')
path_list.append(h5path + 'hw_lp')
path_list.append(h5path + 'hw_ns')
path_list.append(h5path + 'nc_lp')
path_list.append(h5path + 'nc_vt')
print(path_list)


['D:\\seisbench_data\\vcseis\\ak_lp', 'D:\\seisbench_data\\vcseis\\ak_ns', 'D:\\seisbench_data\\vcseis\\ak_vt', 'D:\\seisbench_data\\vcseis\\cas_lp', 'D:\\seisbench_data\\vcseis\\cas_vt', 'D:\\seisbench_data\\vcseis\\hw_vt', 'D:\\seisbench_data\\vcseis\\hw_lp', 'D:\\seisbench_data\\vcseis\\hw_ns', 'D:\\seisbench_data\\vcseis\\nc_lp', 'D:\\seisbench_data\\vcseis\\nc_vt']


In [14]:
cur_data = sbd.WaveformDataset(path_list[0])
print(cur_data.metadata.columns.tolist())
print(cur_data.metadata.loc[0, 'trace_p_arrival_sample'])
print(cur_data.metadata.loc[0, 'trace_s_arrival_sample'])

2025-12-18 18:50:20,048 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


['index', 'source_id', 'source_origin_time', 'source_latitude_deg', 'source_longitude_deg', 'source_depth_km', 'source_magnitude', 'source_magnitude_type', 'source_type', 'station_network_code', 'station_code', 'station_location_code', 'trace_channel', 'station_latitude_deg', 'station_longitude_deg', 'station_elevation_m', 'station_epicentral_distance_m', 'path_azimuth_deg', 'path_back_azimuth_deg', 'trace_p_arrival_time', 'trace_s_arrival_time', 'trace_p_max_weight', 'trace_s_max_weight', 'trace_p_first_motion', 'trace_name', 'trace_sampling_rate_hz', 'trace_has_spikes', 'trace_start_time', 'trace_p_arrival_sample', 'trace_p_status', 'trace_s_arrival_sample', 'trace_s_status', 'trace_snr_db', 'trace_mean_snr_db', 'trace_frequency_index', 'split', 'trace_name_original', 'source_frequency_index', 'source_active_volcano_distance_m', 'trace_chunk', 'trace_component_order']
5999
6166


In [16]:
file_exists=False
counter = 0
for i in range (len(path_list)):
    cur_data = sbd.WaveformDataset(path_list[i])
    print(path_list[i])
    for j in range (len(cur_data)):
        itp = cur_data.metadata.loc[j, 'trace_p_arrival_sample']
        its = cur_data.metadata.loc[j, 'trace_s_arrival_sample']
        station_id = cur_data.metadata.loc[j, 'station_code']
        cur_wave = cur_data.get_waveforms(j).T
        if(cur_wave.shape[0] < 11000):
            continue
        z = cur_wave[:, 0]
        n = cur_wave[:, 1]
        e = cur_wave[:, 2]
        new_wave = np.vstack((e,n,z)).T
        instance_id = f'{i}+{j}'
        save_path = f'D:\\seisbench_data\\vcseis\\npzdata\\{instance_id}.npz'
        np.savez_compressed(
            save_path,
            data = new_wave,        
            itp = itp,
            its = its, 
            channels = 'E_N_U',
            station_id = station_id
        )
        # deal with csv
        
        if i == 0 or i == 3 or i == 6 or i == 8:
            label = 'B'
            csv_path = 'D:\\seisbench_data\\vcseis\\npzdata\\lpdata.csv'
        elif i == 1 or i == 7:
            label = 'Noise'
            csv_path = 'D:\\seisbench_data\\vcseis\\npzdata\\nsdata.csv'
        else:
            label = 'A'
            csv_path = 'D:\\seisbench_data\\vcseis\\npzdata\\vtdata.csv'

        processed_event = {
            'id':counter,
            'fname':f'{instance_id}.npz',
            'itp':itp,
            'its':its,
            'channels':'E_N_U',
            'label': label
            }
        with open(csv_path, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=['id','fname', 'itp', 'its', 'channels','label'])
            
            if not file_exists:
                writer.writeheader()
                file_exists = True # 后续循环不再写表头
            writer.writerow(processed_event)
        counter += 1

        

2025-12-18 18:51:13,530 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


D:\seisbench_data\vcseis\ak_lp


2025-12-18 18:58:04,969 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


D:\seisbench_data\vcseis\ak_ns


2025-12-18 18:58:48,337 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


...

D:\seisbench_data\vcseis\hw_ns


2025-12-18 19:13:35,415 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


D:\seisbench_data\vcseis\nc_lp


2025-12-18 19:14:06,752 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


D:\seisbench_data\vcseis\nc_vt
